# Sign Language MNIST — Algorithm Lab

Train **one algorithm at a time** on the Sign Language MNIST dataset and
compare results with your teammates.

**Runs identically on Mac, Windows, Linux and Chromebook** — the code executes on
Google's servers, not your machine, so everyone gets the same numbers on the same
hardware.

### How to use
1. Run **Step 1** (setup) and **Step 2** (upload the dataset).
2. In **Step 4**, set `MY_ALGORITHM` to the one you were assigned.
3. Run **Step 5** to train, then **Step 6** to download your result.
4. One person collects everyone's `.json` files and runs **Step 7** for the
   combined chart.

**Adding your own algorithm:** edit the registry in Step 3 — one function plus
one line. Nothing else changes.

> Menu: *Runtime → Change runtime type → T4 GPU* makes the neural models much
> faster. The classical models do not use the GPU.

## Step 1 — Setup
Installs the libraries and fixes the random seeds so results are reproducible.

In [ ]:
import sys, os, random, platform, json, time, io, zipfile
from pathlib import Path

# Colab already has these; the install is a no-op there and helps local Jupyter.
try:
    import sklearn, matplotlib, pandas, numpy
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'scikit-learn', 'matplotlib', 'pandas', 'numpy'])

import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    HAS_TORCH = True
    GPU = torch.cuda.is_available()
except ImportError:
    HAS_TORCH, GPU = False, False

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
print(f"Python {platform.python_version()} on {platform.system()}")
print(f"PyTorch available: {HAS_TORCH}   GPU: {GPU}")
print('Setup complete.')

## Step 2 — Get the dataset

Download **archive.zip** from
[Sign Language MNIST on Kaggle](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)
(you need a free Kaggle account), then run the cell below and select the file.

The upload is ~65 MB and takes a minute or two.

In [ ]:
DATA = Path('data'); DATA.mkdir(exist_ok=True)

existing = list(DATA.glob('*.zip')) + list(DATA.rglob('*train*.csv'))
if existing:
    print(f'Dataset already here: {existing[0].name}')
else:
    try:
        from google.colab import files          # running in Colab
        print('Select archive.zip from your computer...')
        up = files.upload()
        for name in up:
            Path(name).rename(DATA / name)
            print('saved to', DATA / name)
    except ImportError:                          # local Jupyter
        print(f'Not in Colab. Put archive.zip (or the CSVs) into: {DATA.resolve()}')

In [ ]:
# Load the data. Pixel values are scaled to 0-1; the official train/test split
# is used exactly as published, so everyone evaluates on the same images.
def find_dataset():
    z = sorted(DATA.glob('*.zip'))
    if z: return z[0]
    if list(DATA.rglob('*train*.csv')): return DATA
    raise SystemExit('No dataset found — run the cell above first.')

def load(path):
    def read(part, src):
        if isinstance(src, zipfile.ZipFile):
            names = [n for n in src.namelist()
                     if part in n.lower() and n.lower().endswith('.csv')]
            with src.open(sorted(names, key=len)[0]) as fh:
                return pd.read_csv(io.BytesIO(fh.read()))
        hits = sorted(src.rglob(f'*{part}*.csv'), key=lambda p: len(p.name))
        return pd.read_csv(hits[0])
    if Path(path).suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as zf:
            tr, te = read('train', zf), read('test', zf)
    else:
        tr, te = read('train', path), read('test', path)
    xy = lambda df: (df.iloc[:, 1:].to_numpy().astype(np.float32) / 255.0,
                     df.iloc[:, 0].to_numpy().astype(np.int64))
    Xtr, ytr = xy(tr); Xte, yte = xy(te)
    return Xtr, ytr, Xte, yte

X_train, y_train, X_test, y_test = load(find_dataset())
LETTER = {i: chr(ord('A') + i) for i in range(26)}
CLASSES = np.unique(np.concatenate([y_train, y_test]))

print(f'train {X_train.shape}   test {X_test.shape}   classes {len(CLASSES)}')
print('letters:', ' '.join(LETTER[int(c)] for c in CLASSES))
print('\n(J and Z are absent — they are signed with motion, so a still image')
print(' cannot represent them.)')

fig, axes = plt.subplots(2, 6, figsize=(9, 3.4))
for ax, c in zip(axes.ravel(), CLASSES[:12]):
    i = int(np.where(y_train == c)[0][0])
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(LETTER[int(c)]); ax.axis('off')
fig.suptitle('Example images'); fig.tight_layout()
fig.savefig(RESULTS / 'samples.png', dpi=130)
from IPython.display import Image, display
display(Image(str(RESULTS / 'samples.png')))

## Step 3 — The algorithm registry

**This is the only cell you edit to add an algorithm.**

Write a function returning any model with `.fit(X, y)` and `.predict(X)`, then
add one line to `ALGORITHMS`. The input `X` is `(n_images, 784)` with pixel
values in 0–1; `y` is the letter class.

In [ ]:
def make_lda():
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    return LinearDiscriminantAnalysis()

def make_svm():
    from sklearn.svm import SVC
    return SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED)

def make_logreg():
    from sklearn.linear_model import LogisticRegression
    return LogisticRegression(max_iter=1000, random_state=SEED)

def make_knn():
    from sklearn.neighbors import KNeighborsClassifier
    return KNeighborsClassifier(n_neighbors=5)

def make_gboost():
    # HistGradientBoosting: plain GradientBoosting trains
    # (classes x estimators) trees and takes hours on 784 features.
    # early_stopping pinned: the 'auto' default holds back 10% of the
    # training data above 10k samples, and whether it does so depends
    # on the sklearn version -- which made Colab and local disagree.
    from sklearn.ensemble import HistGradientBoostingClassifier
    return HistGradientBoostingClassifier(max_iter=150, random_state=SEED,
                                          early_stopping=False)

def make_mlp():
    from sklearn.neural_network import MLPClassifier
    return MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=60,
                         random_state=SEED)

def make_rf():
    from sklearn.ensemble import RandomForestClassifier
    return RandomForestClassifier(n_estimators=300, random_state=SEED)

def make_nb():
    from sklearn.naive_bayes import GaussianNB
    return GaussianNB()

def make_tree():
    from sklearn.tree import DecisionTreeClassifier
    return DecisionTreeClassifier(random_state=SEED)


# ---- neural models (need PyTorch; use a GPU runtime for speed) ----
class TorchModel:
    """Gives PyTorch models the same .fit()/.predict() interface as sklearn."""
    def __init__(self, kind='gru', epochs=25, lr=2e-3, batch=256):
        self.kind, self.epochs, self.lr, self.batch = kind, epochs, lr, batch
    def _shape(self, X):
        return X.reshape(-1, 28, 28) if self.kind == 'gru' else X.reshape(-1, 1, 28, 28)
    def _build(self, n_cls):
        import torch.nn as nn
        if self.kind == 'gru':
            class Net(nn.Module):
                def __init__(s):
                    super().__init__()
                    s.gru = nn.GRU(28, 128, batch_first=True, bidirectional=True)
                    s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(256, n_cls))
                def forward(s, x):
                    o, _ = s.gru(x); return s.head(o.mean(1))
            return Net()
        return nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Linear(128, n_cls))
    def fit(self, X, y):
        import torch, torch.nn as nn
        torch.manual_seed(SEED)
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = self._build(int(y.max()) + 1).to(dev)
        xt = torch.tensor(self._shape(X), dtype=torch.float32, device=dev)
        yt = torch.tensor(y, dtype=torch.long, device=dev)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        for _ in range(self.epochs):
            self.model.train()
            perm = torch.randperm(len(xt), device=dev)
            for i in range(0, len(xt), self.batch):
                idx = perm[i:i+self.batch]
                opt.zero_grad(); crit(self.model(xt[idx]), yt[idx]).backward(); opt.step()
        return self
    def predict(self, X):
        import torch
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.eval(); xe = torch.tensor(self._shape(X), dtype=torch.float32, device=dev)
        out = []
        with torch.no_grad():
            for i in range(0, len(xe), 1024):
                out.append(self.model(xe[i:i+1024]).argmax(-1).cpu().numpy())
        return np.concatenate(out)

make_gru = lambda: TorchModel('gru', epochs=25)
make_cnn = lambda: TorchModel('cnn', epochs=12, lr=1e-3, batch=128)


# =====================================================================
#  ADD YOUR ALGORITHM HERE:  'key': ('Display Name', factory_function)
# =====================================================================
ALGORITHMS = {
    'lda':    ('LDA',                 make_lda),
    'svm':    ('SVM',                 make_svm),
    'logreg': ('Logistic Regression', make_logreg),
    'knn':    ('k-NN',                make_knn),
    'gboost': ('Gradient Boosting',   make_gboost),
    'mlp':    ('MLP',                 make_mlp),
    'rf':     ('Random Forest',       make_rf),
    'nb':     ('Naive Bayes',         make_nb),
    'tree':   ('Decision Tree',       make_tree),
    'gru':    ('GRU (recurrent)',     make_gru),
    'cnn':    ('CNN (convolutional)', make_cnn),
}

print('Available algorithms:')
for k, (name, _) in ALGORITHMS.items():
    print(f'  {k:<9}{name}')

## Step 4 — Choose your algorithm

Set this to the algorithm you were assigned, then run the next cell.

> **Leave `TRAIN_SIZE` at 0** unless you are just testing. Any other value
> trains on a subset, which lowers accuracy and makes your number **not
> comparable** with teammates who used the full dataset. If you do change it,
> say so in your report.


In [ ]:
MY_ALGORITHM = 'lda'  #@param ['lda','svm','logreg','knn','gboost','mlp','rf','nb','tree','gru','cnn']

# 0 = use all 27,455 training images. KEEP THIS AT 0 for comparable results.
TRAIN_SIZE = 0  #@param {type:"integer"}

print(f'Selected: {ALGORITHMS[MY_ALGORITHM][0]}')
if TRAIN_SIZE:
    print(f'\n*** WARNING: training on only {TRAIN_SIZE:,} of {len(X_train):,} images.')
    print('    Your accuracy will be LOWER and NOT comparable with teammates')
    print('    who used the full set. Set TRAIN_SIZE = 0 for the real run. ***')
else:
    print(f'Training on all {len(X_train):,} images (comparable with everyone else).')


## Step 5 — Train and evaluate

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

name, model = ALGORITHMS[MY_ALGORITHM][0], ALGORITHMS[MY_ALGORITHM][1]()

Xtr, ytr = X_train, y_train
if TRAIN_SIZE and TRAIN_SIZE < len(Xtr):
    idx = np.random.default_rng(SEED).choice(len(Xtr), TRAIN_SIZE, replace=False)
    Xtr, ytr = Xtr[idx], ytr[idx]

print(f'Training {name} on {len(Xtr):,} images...')
t0 = time.time(); model.fit(Xtr, ytr); train_s = time.time() - t0
t0 = time.time(); pred = model.predict(X_test); pred_s = time.time() - t0

acc = accuracy_score(y_test, pred) * 100
f1 = f1_score(y_test, pred, average='macro') * 100
print(f'\n{name}')
print(f'  accuracy : {acc:.2f}%')
print(f'  macro F1 : {f1:.2f}%')
print(f'  training : {train_s:.1f}s      prediction: {pred_s:.1f}s')

result = {
    'algo': MY_ALGORITHM, 'name': name,
    'accuracy': float(acc), 'macro_f1': float(f1),
    'train_seconds': round(train_s, 1), 'predict_seconds': round(pred_s, 1),
    'n_train': int(len(Xtr)), 'n_test': int(len(X_test)),
    'n_classes': int(len(CLASSES)),
    'classes': [LETTER[int(c)] for c in CLASSES],
    'confusion': confusion_matrix(y_test, pred, labels=CLASSES).tolist(),
    'gpu': bool(GPU), 'when': time.strftime('%Y-%m-%d %H:%M'),
}
out = RESULTS / f'{MY_ALGORITHM}.json'
out.write_text(json.dumps(result, indent=2))
print(f'\nsaved -> {out}')

In [ ]:
# Per-letter detail and the confusion matrix
print(classification_report(y_test, pred,
                            labels=CLASSES,
                            target_names=[LETTER[int(c)] for c in CLASSES],
                            digits=3, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=CLASSES).astype(float)
row = cm.sum(1, keepdims=True); row[row == 0] = 1
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm / row, cmap='Blues', vmin=0, vmax=1)
names = [LETTER[int(c)] for c in CLASSES]
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, fontsize=8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion matrix — {name}')
fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout()
fig.savefig(RESULTS / f'confusion_{MY_ALGORITHM}.png', dpi=130)
display(Image(str(RESULTS / f'confusion_{MY_ALGORITHM}.png')))

## Step 5b — (optional) Train several algorithms at once

Step 5 trains the single algorithm you picked. Use this cell instead to run
**several, or all of them, in one go**.

### How long things take

Colab's free tier gives roughly **2 CPU cores**, so it is much slower than a
desktop. Rough guide for all 27,455 images:

| algorithm | on Colab (free CPU) |
|---|---|
| `lda`, `knn`, `nb`, `tree` | seconds |
| `rf` | 1-2 minutes |
| `svm`, `gboost` | 4-8 minutes |
| `logreg` | 5-10 minutes |
| `mlp` | 10-20 minutes (the slowest) |
| `gru`, `cnn` | fast **with a GPU runtime**, slow without |

> **The GPU only helps `gru` and `cnn`.** Every other algorithm here is
> scikit-learn, which is CPU-only — switching runtime type will not speed them
> up at all.

> If something seems stuck, it is almost certainly just slow. Do **not** lower
> `TRAIN_SIZE` or change model settings to speed it up: the result would no
> longer be comparable with your teammates'.


In [ ]:
# Which algorithms to train. Use list(ALGORITHMS) for every one of them.
TO_RUN = ['lda', 'knn', 'rf']          # e.g. ['svm','rf'] or list(ALGORITHMS)

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

Xtr_, ytr_ = X_train, y_train
if TRAIN_SIZE and TRAIN_SIZE < len(Xtr_):
    _idx = np.random.default_rng(SEED).choice(len(Xtr_), TRAIN_SIZE, replace=False)
    Xtr_, ytr_ = Xtr_[_idx], ytr_[_idx]

for key in TO_RUN:
    if key not in ALGORITHMS:
        print(f'  ?  unknown algorithm: {key}'); continue
    label, factory = ALGORITHMS[key]
    print(f'training {label} on {len(Xtr_):,} images...', flush=True)
    try:
        mdl = factory()
        t0 = time.time(); mdl.fit(Xtr_, ytr_); tr_s = time.time() - t0
        p = mdl.predict(X_test)
    except Exception as e:
        print(f'  FAILED ({type(e).__name__}: {e})')
        if 'torch' in str(e).lower():
            print('  -> needs PyTorch; Runtime > Change runtime type > T4 GPU')
        continue
    a_ = accuracy_score(y_test, p) * 100
    f_ = f1_score(y_test, p, average='macro') * 100
    (RESULTS / f'{key}.json').write_text(json.dumps({
        'algo': key, 'name': label, 'accuracy': float(a_), 'macro_f1': float(f_),
        'train_seconds': round(tr_s, 1), 'predict_seconds': 0.0,
        'n_train': int(len(Xtr_)), 'n_test': int(len(X_test)),
        'n_classes': int(len(CLASSES)),
        'classes': [LETTER[int(c)] for c in CLASSES],
        'confusion': confusion_matrix(y_test, p, labels=CLASSES).tolist(),
        'gpu': bool(GPU), 'when': time.strftime('%Y-%m-%d %H:%M')}, indent=2))
    print(f'  {label}: accuracy {a_:.2f}%   macro-F1 {f_:.2f}%   ({tr_s:.1f}s)\n')

print('Done. Run Step 7 to build the comparison chart.')

## Step 6 — Download your result

Send the downloaded `.json` to whoever is assembling the final report.

In [ ]:
try:
    from google.colab import files
    files.download(str(RESULTS / f'{MY_ALGORITHM}.json'))
    files.download(str(RESULTS / f'confusion_{MY_ALGORITHM}.png'))
except ImportError:
    print(f'Not in Colab — your files are in: {RESULTS.resolve()}')

## Step 7 — Combine everyone's results

**Only the person assembling the report needs this.** Upload every teammate's
`.json` file, then run the cell — it produces the comparison chart and a table.

In [ ]:
try:
    from google.colab import files
    print('Select all the .json result files from your teammates...')
    up = files.upload()
    for n in up:
        Path(n).rename(RESULTS / n)
except ImportError:
    print(f'Not in Colab — put the .json files into {RESULTS.resolve()}')

runs = []
for f in sorted(RESULTS.glob('*.json')):
    if f.name == 'summary.json':
        continue
    try:
        runs.append(json.loads(f.read_text()))
    except json.JSONDecodeError:
        print('skipping unreadable', f.name)

runs.sort(key=lambda r: r['accuracy'], reverse=True)
names = [r['name'] for r in runs]
acc = [r['accuracy']/100 for r in runs]
f1s = [r['macro_f1']/100 for r in runs]

x = np.arange(len(runs)); w = 0.38
fig, ax = plt.subplots(figsize=(max(8, 1.4*len(runs)), 5.5))
ax.bar(x - w/2, acc, w, label='Accuracy', color='#4472C4')
ax.bar(x + w/2, f1s, w, label='Macro F1', color='#ED7D31')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.set_title('Sign Language MNIST: algorithm comparison\n'
             f"{runs[0]['n_classes']} letters · {runs[0]['n_train']:,} train / "
             f"{runs[0]['n_test']:,} test images")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.legend(); ax.grid(axis='y', alpha=0.25)
for i, v in enumerate(acc):
    ax.text(i - w/2, v + 0.012, f'{v*100:.1f}', ha='center', fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS / 'comparison_chart.png', dpi=150)
display(Image(str(RESULTS / 'comparison_chart.png')))

print(f"\n{'algorithm':<24}{'accuracy':>10}{'macro F1':>10}{'train s':>10}")
print('-'*54)
for r in runs:
    print(f"{r['name']:<24}{r['accuracy']:>9.2f}%{r['macro_f1']:>9.2f}%"
          f"{r['train_seconds']:>10.1f}")

pd.DataFrame([{'Algorithm': r['name'], 'Accuracy (%)': round(r['accuracy'], 2),
               'Macro F1 (%)': round(r['macro_f1'], 2),
               'Train (s)': r['train_seconds']} for r in runs]
            ).to_csv(RESULTS / 'summary.csv', index=False)
print('\ntable saved -> results/summary.csv')

---
### Notes for the report

- The **official train/test split** is used unchanged, so every algorithm is
  evaluated on exactly the same 7,172 images.
- **Macro F1** averages the score of every letter equally, so a rare letter
  counts as much as a common one. Accuracy alone can hide poor performance on
  infrequent classes.
- Every classical model here treats the 784 pixels as **independent features** —
  none of them uses the 2D layout of the image. The CNN does, which is why it
  performs noticeably better and is worth including as a contrast.
- **J** and **Z** are missing from the dataset because they involve motion.
  A still-image model cannot represent them at all — a real limitation of this
  approach to sign language, worth a sentence in any write-up.